# 01. Cochains, exactness, and the vector obstruction

**Level:** Novice to intermediate

**Prerequisites:** Lesson 000 and familiarity with a graph's vertices and edges.

**Canonical claim anchors:** `WDP-CH04-P02`, `WDP-CH04-C01`, `WDP-CH04-P07`, `WDP-CH07-P02`

## Learning objectives

- compute exact 0- and 1-coboundaries on a finite complex
- separate closedness from exactness and read a refusal witness
- certify a quotient defect exactly with a repair and a bounded circulation

Work top-to-bottom. Predict each worked result before executing it, then use
the exercise to test whether you can separate the finite result from its scope
boundary.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next((candidate for candidate in (start, *start.parents)
             if (candidate / 'pyproject.toml').is_file()
             and (candidate / 'code' / 'witnessed_difference').is_dir()), None)
if root is None:
    raise RuntimeError('could not locate the self-contained paper package')
sys.path.insert(0, str(root / 'code'))


## Idea: local differences can fail to come from global labels

A 0-cochain assigns a value to each vertex. Its coboundary `d0` assigns the
difference of endpoint values to every oriented edge. A 1-cochain is
**exact** when it arises this way. On each admitted triangular face, `d1`
adds the oriented edge values around the boundary. Exact cochains are closed
because successive differences telescope.

The implementation keeps rational arithmetic exact. It also returns a typed
refusal when a cycle inconsistency prevents reconstruction of a potential.


## Visual checkpoint

This lesson uses printed exact data rather than a diagram. Read each returned
status and refusal as part of the result; do not reduce the cell to its final
number.


## Derivation

For a face `(a,b,c)`, the convention is

`d1(alpha) = alpha(a,b) + alpha(b,c) - alpha(a,c)`.

If `alpha=d0(phi)`, substitution gives
`(phi_b-phi_a)+(phi_c-phi_b)-(phi_c-phi_a)=0`.

The vector example uses three two-coordinate edge differences. Their cycle
sum is invariant under adding a vertex potential. Routing the entire residual
to one edge reaches both triangle-inequality lower bounds used by the finite
example.


In [ ]:
from witnessed_difference.cochains import FiniteComplex, d0, d1, solve_potential
from witnessed_difference.notebook_support import display_json
from witnessed_difference.vector_obstruction import source_three_node_example

triangle = FiniteComplex(('a', 'b', 'c'), (('a', 'b'), ('b', 'c'), ('a', 'c')), (('a', 'b', 'c'),))
potential = {'a': 0, 'b': 1, 'c': 3}
gradient = d0(triangle, potential)
print(display_json({'d0': {f'{u}->{v}': value for (u, v), value in gradient.items()}}))
print(display_json({'d1_d0': {','.join(face): value for face, value in d1(triangle, gradient).items()}}))
print(display_json({'path_independence': {
    'direct_a_to_c': gradient[('a', 'c')],
    'via_b_a_to_c': gradient[('a', 'b')] + gradient[('b', 'c')],
}}))
print(display_json(source_three_node_example()))


## Read the result

`d1_d0` vanishes exactly. The vector replay reports circulation `(1,1)`, an
L1-of-L1 minimum of `2`, and squared L1-of-L2 minimum `2`; `1.414214` is only
a six-decimal presentation of the latter square root. The exact certificate,
not the decimal, is the replayable object.


In [ ]:
from witnessed_difference.cochains import finite_quotient_l1
from witnessed_difference.outcomes import Refused

inconsistent = {('a', 'b'): 1, ('b', 'c'): 1, ('a', 'c'): 3}
reconstruction = solve_potential(triangle, inconsistent)
print(display_json(reconstruction))
assert isinstance(reconstruction, Refused)
finite_candidates = (
    {'a': 0, 'b': 0, 'c': 0},
    {'a': 0, 'b': 1, 'c': 2},
    {'a': 0, 'b': 1, 'c': 3},
)
print(display_json({'declared_finite_quotient_minimizer': finite_quotient_l1(
    triangle, inconsistent, finite_candidates,
)}))
print('A nonzero cycle sum blocks a global potential on this admitted graph.')

from witnessed_difference.obstruction import obstruction_signature, promote_diagnostic

# WDP-CH04-P07: an exact LP returns a repair and a circulation; both are
# re-checked independently before the value is reported.
signature = obstruction_signature(triangle, inconsistent, closure_declared=True).value
certificate = signature.certificate
print(display_json({'obstruction_signature': {
    'phi1': signature.phi1,
    'gamma2': signature.gamma2,
    'closure_ratio': signature.closure_ratio,
    'diagnostic_class': signature.diagnostic_class.value,
    'repair_potential': certificate.potential,
    'dual_circulation': certificate.dual,
}}))
print(display_json(promote_diagnostic(signature, 'ExactRepair')))


## Exercise

Change the inconsistent value on edge `(a,c)` from `3` to `2`. Predict whether
`solve_potential` will accept it, then recover the returned potential. Which
freedom remains in that potential?


In [ ]:
consistent = {('a', 'b'): 1, ('b', 'c'): 1, ('a', 'c'): 2}
recovered = solve_potential(triangle, consistent)
print(display_json(recovered))
print('Adding the same constant to every vertex leaves d0 unchanged.')


## Scope check

These are exact results on explicit finite complexes and vectors. They
do not establish a continuum Hodge theorem or a physical field equation.
**AuthorityEffect: None.**


## Continue the learning path

- Previous: [Primitive difference and finite symmetry](000_primitive_difference.ipynb)
- Index: [Notebook curriculum](README.md)
- Next: [Retention, re-access, and witnessed difference](020_retention_and_reaccess.ipynb)
